# Step 7 — Verify & Fix: regenerate every statistic from the FULL 120-trial dataset

Your recordings folder holds all **120 experiment trials** (90 Phase 1 + 30 studio) plus 9 pilot recordings — but the published analysis only ever listed 88 files and skipped all 15 `kbaby` and all 15 `ccar` trials. This notebook analyzes **everything on disk** and replaces the placeholder statistics with real computations.

**It produces (in `results/`):**
- Real per-fold F1s + a **real bootstrap 95% CI** for macro F1 (Sect. 3.7)
- `zeroshot_baseline.json` — real zero-shot baseline per fold and per tier (Table 3) + a **real Wilcoxon test**
- `phase1_trials.csv`, `studio_trials.csv` — per-trial localization for all 120 recordings
- `localization_stats.json` — per-angle + overall MAE/within-15, both phases (Table 5)
- `recording_manifest.json` — found vs. design (should be 120/120), pilots listed separately
- `final_numbers.json` — everything the manuscript needs

**Run notebooks 01–02 first** (needs `models/embeddings_cache.npz` and the PANNs checkpoint), then Run All. Part 3 takes ~15–30 min for 120 VRS files. Send the whole `results/` folder back when done.

In [ ]:
from pathlib import Path

# ── Base path Set your base path here ─────────────────
BASE = Path('/path/to/your/esas_project')  # <-- set this
# ────────────────────────────────────────────────────────────

RECORDINGS  = BASE / 'recordings'
ESC50_DIR   = BASE / 'ESC-50'
PANNS_CKPT  = BASE / 'panns_data' / 'Cnn14_mAP=0.431.pth'
MODELS_DIR  = Path('models')
RESULTS_DIR = Path('results')
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
n_vrs = len(list(RECORDINGS.glob('*.vrs'))) if RECORDINGS.exists() else 0
print(f'Recordings dir: {RECORDINGS.exists()}   .vrs files found: {n_vrs} (expect 129)')
print(f'ESC-50: {ESC50_DIR.exists()}   PANNs ckpt: {PANNS_CKPT.exists()}')

In [ ]:
import csv, glob, json, math, re, warnings
from collections import Counter, defaultdict
import numpy as np
from scipy import stats as sps
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_recall_fscore_support, f1_score
warnings.filterwarnings('ignore')

HIGH   = {'car_horn','chainsaw','crackling_fire','glass_breaking','hand_saw','siren','fireworks'}
MEDIUM = {'clock_alarm','clock_tick','crying_baby','dog','door_wood_knock','sneezing',
          'coughing','church_bells','vacuum_cleaner','washing_machine','toilet_flush','cat'}
def get_priority(label):
    if label in HIGH:   return 'HIGH'
    if label in MEDIUM: return 'MEDIUM'
    return 'LOW'

meta = {}
with open(ESC50_DIR / 'meta' / 'esc50.csv') as f:
    for row in csv.DictReader(f):
        meta[row['filename']] = {'cat': row['category'], 'fold': int(row['fold'])}
print(f'ESC-50 metadata: {len(meta)} clips')

## Part 1 — Real 5-fold CV predictions and a REAL bootstrap CI for macro F1

The old notebook drew synthetic normal noise around the point estimate. Here we resample the actual pooled cross-validation predictions.

In [ ]:
CACHE = MODELS_DIR / 'embeddings_cache.npz'
assert CACHE.exists(), 'Run notebook 02 first to create models/embeddings_cache.npz'
d = np.load(CACHE, allow_pickle=True)
X, y, folds = d['X'], d['y'], d['folds']
le = LabelEncoder(); ye = le.fit_transform(y)
classes = list(le.classes_)
print(f'Embeddings: {X.shape}  classes: {classes}')

all_true, all_pred, fine_fold_f1 = [], [], []
for fold in range(1, 6):
    tr, te = folds != fold, folds == fold
    clf = LogisticRegression(C=1.0, max_iter=1000, random_state=42, class_weight='balanced')
    clf.fit(X[tr], ye[tr])
    preds = clf.predict(X[te])
    all_true.extend(ye[te]); all_pred.extend(preds)
    fine_fold_f1.append(float(f1_score(ye[te], preds, average='macro')))
    print(f'  Fold {fold}: macro F1 = {fine_fold_f1[-1]:.4f}')
all_true = np.array(all_true); all_pred = np.array(all_pred)

p_, r_, f_, _ = precision_recall_fscore_support(all_true, all_pred, labels=range(len(classes)), zero_division=0)
_, _, mf, _   = precision_recall_fscore_support(all_true, all_pred, average='macro', zero_division=0)
per_tier_fine = {c: {'P': round(float(p_[i]),3), 'R': round(float(r_[i]),3), 'F1': round(float(f_[i]),3)} for i,c in enumerate(classes)}
print(f'\nPooled macro F1 = {mf:.3f}  (Table 2 macro row)')

# REAL bootstrap: resample pooled prediction pairs, recompute macro F1
rng  = np.random.default_rng(42)
n    = len(all_true)
boot = []
for _ in range(1000):
    idx = rng.integers(0, n, n)
    boot.append(f1_score(all_true[idx], all_pred[idx], average='macro'))
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f'REAL bootstrap 95% CI for macro F1: {mf:.3f} ({lo:.3f}-{hi:.3f})   <-- use this in Sect. 3.7')
np.savez(RESULTS_DIR / 'cv_predictions.npz', all_true=all_true, all_pred=all_pred,
         classes=np.array(classes), fine_fold_f1=np.array(fine_fold_f1))
f1_ci = {'macro_f1': round(float(mf),3), 'ci_lo': round(float(lo),3), 'ci_hi': round(float(hi),3),
         'fine_fold_f1': [round(v,4) for v in fine_fold_f1], 'per_tier': per_tier_fine}

## Part 2 — Real zero-shot baseline + real Wilcoxon test

The paper's baseline is "string-matching between PANNs class names and hazard labels" — but no notebook ever computed it. This cell computes it: PANNs clipwise argmax → AudioSet label → keyword match to HIGH/MEDIUM/LOW. Takes ~15 min on CPU the first time; cached afterwards.

**Whatever numbers come out here are the paper's numbers** — for Table 3's baseline column and the Wilcoxon test — even if they differ from the old 0.706 / W=15 / p=0.031.

In [ ]:
import librosa
from panns_inference import AudioTagging, labels as audioset_labels

ZS_CACHE = MODELS_DIR / 'zeroshot_argmax_cache.npz'
if ZS_CACHE.exists():
    z = np.load(ZS_CACHE, allow_pickle=True)
    zs_files, zs_argmax = list(z['files']), z['argmax']
    print(f'Loaded cached zero-shot argmax for {len(zs_files)} clips')
else:
    panns = AudioTagging(checkpoint_path=str(PANNS_CKPT), device='cpu')
    files = sorted(glob.glob(str(ESC50_DIR / 'audio' / '*.wav')))
    zs_files, zs_argmax = [], []
    for i, path in enumerate(files):
        fn = Path(path).name
        if fn not in meta: continue
        audio, _ = librosa.load(path, sr=32000, mono=True)
        audio = audio[:64000] if len(audio) >= 64000 else np.pad(audio, (0, 64000-len(audio)))
        clipwise, _ = panns.inference(audio[None, :])
        zs_files.append(fn); zs_argmax.append(int(np.argmax(clipwise[0])))
        if (i+1) % 400 == 0: print(f'  {i+1}/{len(files)}')
    zs_argmax = np.array(zs_argmax)
    np.savez(ZS_CACHE, files=np.array(zs_files), argmax=zs_argmax)
    print(f'Cached: {ZS_CACHE}')

HIGH_KW   = ['horn','honk','chainsaw','fire','glass','saw','siren','firework','explosion','gunshot']
MEDIUM_KW = ['alarm','clock','tick','baby','infant','cry','dog','bark','knock','sneez','cough',
             'bell','vacuum','washing','toilet','cat','meow','purr','telephone','ring']
def label_to_tier(name):
    s = name.lower()
    if any(k in s for k in HIGH_KW):   return 'HIGH'
    if any(k in s for k in MEDIUM_KW): return 'MEDIUM'
    return 'LOW'

true_t = np.array([get_priority(meta[f]['cat']) for f in zs_files])
pred_t = np.array([label_to_tier(audioset_labels[a]) for a in zs_argmax])
fold_a = np.array([meta[f]['fold'] for f in zs_files])

tiers = ['HIGH','MEDIUM','LOW']
bp, br, bf, _ = precision_recall_fscore_support(true_t, pred_t, labels=tiers, zero_division=0)
_, _, bmf, _  = precision_recall_fscore_support(true_t, pred_t, average='macro', zero_division=0)
base_fold_f1 = [float(f1_score(true_t[fold_a==k], pred_t[fold_a==k], average='macro')) for k in range(1,6)]
print('Zero-shot baseline per tier (Table 3 baseline column):')
for i,t in enumerate(tiers): print(f'  {t:<7} F1 = {bf[i]:.3f}')
print(f'  Macro F1 = {bmf:.3f}')
print(f'  Per-fold: {[round(v,4) for v in base_fold_f1]}')

W, pval = sps.wilcoxon(np.array(fine_fold_f1), np.array(base_fold_f1), alternative='greater')
nz = np.sum(np.array(fine_fold_f1) != np.array(base_fold_f1))
print(f'\nREAL Wilcoxon: W = {W:.1f}, p = {pval:.5f} (one-sided, n={nz} pairs)  <-- use this in Sect. 3.7')
zs = {'macro_f1': round(float(bmf),3), 'per_tier_f1': {t: round(float(bf[i]),3) for i,t in enumerate(tiers)},
      'per_fold_f1': [round(v,4) for v in base_fold_f1], 'wilcoxon_W': float(W), 'wilcoxon_p': round(float(pval),5)}
json.dump(zs, open(RESULTS_DIR / 'zeroshot_baseline.json','w'), indent=2)
print(f"Saved: {RESULTS_DIR/'zeroshot_baseline.json'}")

## Part 3 — Localization from VRS: ALL 120 trials, both phases

Same pipeline as notebook 03 (onset detection, 50 ms skip, 1.5 s direct-path window, 6-pair GCC-PHAT median), but files are discovered from disk and the parser now covers **all four sounds**: fire alarm, telephone ring, **crying baby** (`kbaby…`, kitchen), and **car horn** (`ccar_…`, traffic). The 9 pilot recordings (`Fire0…`, `Horn0…`, `Ofire0_4`) are excluded and listed in the manifest.

In [ ]:
SR, D, C = 48_000, 0.060, 343.0
def gcc_phat(s1, s2):
    n   = 2*int(2**math.ceil(math.log2(max(len(s1),len(s2)))))
    X1  = np.fft.rfft(s1, n=n); X2 = np.fft.rfft(s2, n=n)
    cc  = X1*np.conj(X2)
    gcc = np.fft.irfft(cc/(np.abs(cc)+1e-10), n=n)
    ml  = int(SR*D/C)
    gh  = np.concatenate([gcc[-ml:], gcc[:ml+1]])
    pk  = int(np.argmax(gh)) - ml
    return math.degrees(math.asin(np.clip(pk/SR*C/D,-1,1)))

from projectaria_tools.core import data_provider as dp
def analyse_vrs(vrs_path):
    provider = dp.create_vrs_data_provider(str(vrs_path))
    audio_id = provider.get_stream_id_from_label('mic')
    chunks = []
    for i in range(provider.get_num_data(audio_id)):
        frame,_ = provider.get_audio_data_by_index(audio_id, i)
        try:    raw = np.array(frame.data, dtype=np.float32)
        except: raw = np.array(frame.audio_array, dtype=np.float32)
        if raw.ndim==1 and len(raw)%7==0: chunks.append(raw.reshape(7,-1))
    if not chunks: return None
    audio = np.concatenate(chunks, axis=1)
    mono  = audio[0]
    rms   = np.array([np.sqrt(np.mean(mono[i:i+512]**2)) for i in range(0,len(mono)-512,512)])
    bg    = float(np.median(np.sort(rms)[:max(1,len(rms)//3)]))
    onset = next((i*512 for i,r in enumerate(rms) if r>max(bg*5,0.005)), 0)
    s = onset + int(SR*0.05); e = s + int(SR*1.5)
    w = audio[:, s:min(e,audio.shape[1])]
    ang = [gcc_phat(w[c1].astype(np.float32), w[c2].astype(np.float32))
           for c1,c2 in [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]]
    return float(np.median(ang))

# ── Filename parsing: all four sounds ──
PATTERNS = [
    (re.compile(r'^(o|k)(fire|phon)(-?\d+)_(\d+)\.vrs$'),
     lambda m: ({'o':'office','k':'kitchen'}[m.group(1)],
                {'fire':'fire_alarm','phon':'phone_ring'}[m.group(2)],
                int(m.group(3)), int(m.group(4)), 'phase1')),
    (re.compile(r'^kbaby(-?\d+)_(\d+)\.vrs$'),
     lambda m: ('kitchen', 'crying_baby', int(m.group(1)), int(m.group(2)), 'phase1')),
    (re.compile(r'^ccar_(-?\d+)_(\d+)\.vrs$'),
     lambda m: ('traffic', 'car_horn', int(m.group(1)), int(m.group(2)), 'phase1')),
    (re.compile(r'^studio_(fire|phone)(-?\d+)_(\d+)\.vrs$'),
     lambda m: ('studio', {'fire':'fire_alarm','phone':'phone_ring'}[m.group(1)],
                int(m.group(2)), int(m.group(3)), 'studio')),
]
PILOT = re.compile(r'^(Fire|Horn|Ofire)0(_\d+)?\.vrs$')

p1_rows, st_rows, pilots, unmatched = [], [], [], []
for f in sorted(RECORDINGS.glob('*.vrs')):
    if PILOT.match(f.name): pilots.append(f.name); continue
    parsed = None
    for pat, fn in PATTERNS:
        m = pat.match(f.name)
        if m: parsed = fn(m); break
    if parsed is None: unmatched.append(f.name); continue
    amb, snd, ang, tr, phase = parsed
    try:
        est = analyse_vrs(f)
    except Exception as ex:
        print(f'  ERROR {f.name}: {ex}'); continue
    if est is None:
        print(f'  NO AUDIO {f.name}'); continue
    err = abs(est - ang)
    row = {'file':f.name,'ambience':amb,'true_angle':ang,'est_angle':round(est,1),
           'ang_error':round(err,1),'within_15':err<=15,'sound':snd,'trial':tr}
    (p1_rows if phase=='phase1' else st_rows).append(row)
    print(f'  {f.name:<26} {amb:<8} {snd:<12} true={ang:>4}  est={est:+6.1f}  err={err:5.1f}')

print(f'\nAnalysed: {len(p1_rows)} Phase-1 + {len(st_rows)} studio = {len(p1_rows)+len(st_rows)} trials')
print(f'Pilots excluded ({len(pilots)}): {pilots}')
if unmatched: print(f'UNMATCHED FILENAMES — tell Claude about these: {unmatched}')

def write_csv(rows, name):
    if not rows: print(f'  No rows for {name}'); return
    with open(RESULTS_DIR / name, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    print(f'Saved: {RESULTS_DIR/name}  ({len(rows)} trials)')
write_csv(p1_rows, 'phase1_trials.csv'); write_csv(st_rows, 'studio_trials.csv')

def summarize(rows, label):
    if not rows: return None
    errs = np.array([r['ang_error'] for r in rows])
    out = {'n': len(rows), 'MAE': round(float(errs.mean()),1), 'std': round(float(errs.std()),1),
           'within15_pct': round(100*float(np.mean(errs<=15)),1), 'per_angle': {}, 'per_sound': {}}
    by_a, by_s = defaultdict(list), defaultdict(list)
    for r in rows:
        by_a[abs(r['true_angle'])].append(r['ang_error']); by_s[r['sound']].append(r['ang_error'])
    for a in sorted(by_a):
        e = np.array(by_a[a])
        out['per_angle'][str(a)] = {'n': len(e), 'MAE': round(float(e.mean()),1),
                                    'within15_pct': round(100*float(np.mean(e<=15)),1)}
    for s in sorted(by_s):
        e = np.array(by_s[s])
        out['per_sound'][s] = {'n': len(e), 'MAE': round(float(e.mean()),1)}
    print(f"{label}: n={out['n']}  MAE={out['MAE']} ± {out['std']}  within15={out['within15_pct']}%")
    print(f'  per angle: {out["per_angle"]}')
    return out

loc = {'phase1': summarize(p1_rows,'Phase 1'), 'studio': summarize(st_rows,'Phase 2 studio')}
json.dump(loc, open(RESULTS_DIR / 'localization_stats.json','w'), indent=2)

ANGLES, TRIALS = [0,45,-45,90,-90], (1,2,3)
design  = {f'{p}{s}{a}_{t}.vrs' for p in 'ok' for s in ['fire','phon'] for a in ANGLES for t in TRIALS}
design |= {f'kbaby{a}_{t}.vrs' for a in ANGLES for t in TRIALS}
design |= {f'ccar_{a}_{t}.vrs' for a in ANGLES for t in TRIALS}
design |= {f'studio_{s}{a}_{t}.vrs' for s in ['fire','phone'] for a in ANGLES for t in TRIALS}
have = {f.name for f in RECORDINGS.glob('*.vrs')}
manifest = {'design_total': len(design), 'found': len(design & have),
            'missing': sorted(design - have), 'pilots_excluded': pilots,
            'unmatched': unmatched,
            'phase1_analysed': len(p1_rows), 'studio_analysed': len(st_rows)}
json.dump(manifest, open(RESULTS_DIR / 'recording_manifest.json','w'), indent=2)
print(f'\nManifest: {manifest["found"]}/{manifest["design_total"]} design files found; missing: {manifest["missing"]}')

## Part 4 — Final numbers for the manuscript

In [ ]:
frontal_p1 = [r['ang_error'] for r in p1_rows if r['true_angle']==0]
fr = np.array(frontal_p1) if frontal_p1 else None
if fr is not None and len(fr):
    rng = np.random.default_rng(42)
    fboot = [np.mean(rng.choice(fr, size=len(fr), replace=True)) for _ in range(1000)]
    flo, fhi = np.percentile(fboot, [2.5, 97.5])
    frontal = {'n': int(len(fr)), 'MAE': round(float(fr.mean()),1),
               'ci': [round(float(flo),1), round(float(fhi),1)],
               'within15_pct': round(100*float(np.mean(fr<=15)),1)}
else:
    frontal = None

final = {'detection': f1_ci, 'zeroshot_baseline': zs,
         'localization': loc, 'frontal_phase1': frontal, 'manifest': manifest}
json.dump(final, open(RESULTS_DIR / 'final_numbers.json','w'), indent=2)
print(json.dumps(final, indent=2)[:2500])
print('\nDone. Send the whole results/ folder back for the manuscript update.')